<a href="https://colab.research.google.com/github/archil-09/Transformer_from_scratch/blob/main/andhrej_karapathy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-24 08:59:09--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-09-24 08:59:09 (21.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [5]:
with open(r'input.txt','r',encoding='utf-8') as f:
       text = f.read()

In [6]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [7]:
print(len(text))

1115394


In [8]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [9]:
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [10]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]
print(''.join(decode(encode("hello bacho"))))

hello bacho


In [11]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [12]:
n = int(len(data)*0.9)
train_data = data[:n]
val_data = data[n:]

In [13]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target is {target}")

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [15]:
batch_size = 4
block_size = 8
torch.manual_seed(200)

def get_batch(split):
  data = train_data if split=="train" else val_data
  ix = torch.randint(len(data)-block_size , (batch_size,))
  x=torch.stack([data[i:block_size+i] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb,yb=get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('outputs')
print(yb.shape)
print(yb)

inputs
torch.Size([4, 8])
tensor([[ 1, 42, 39, 59, 45, 46, 58, 43],
        [56, 44, 59, 50,  1, 39, 56, 51],
        [53, 59, 56,  1, 42, 39, 59, 45],
        [ 1, 45, 53, 53, 42, 50, 63,  1]])
outputs
torch.Size([4, 8])
tensor([[42, 39, 59, 45, 46, 58, 43, 56],
        [44, 59, 50,  1, 39, 56, 51, 11],
        [59, 56,  1, 42, 39, 59, 45, 46],
        [45, 53, 53, 42, 50, 63,  1, 57]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(200)

class biagramModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
